In [10]:
# Install required libraries for RAG pipeline
!pip install langchain langchain-community langchain-core
!pip install chromadb
!pip install sentence-transformers
!pip install pypdf2
!pip install google-generativeai
!pip install tiktoken
!pip install langchain-text-splitters
!pip install requests
!pip install google-genai
!pip install -U `langchain-huggingface

/bin/bash: -c: line 1: unexpected EOF while looking for matching ``'
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [12]:
import sys
!{sys.executable} -m pip install -U 'langchain-huggingface'

# Importing all the libraries required
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import SentenceTransformersTokenTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import GooglePalm
from langchain_classic.chains import RetrievalQA
import chromadb
import requests
import os
import tempfile

In [5]:
# Extraction of text from PDF file
def extract_pdf_content(pdf_path):
  text = ""
  # Download the PDF if the path is a URL
  if pdf_path.startswith("http://") or pdf_path.startswith("https://"):
    response = requests.get(pdf_path)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    # Create a temporary file to save the PDF content
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_pdf:
      temp_pdf.write(response.content)
      temp_pdf_path = temp_pdf.name

    try:
      with open(temp_pdf_path, "rb") as pdf_file:
        pdf_reader = PdfReader(pdf_file)
        for page in pdf_reader.pages:
          text += page.extract_text()
    finally:
      os.remove(temp_pdf_path) # Clean up the temporary file
  else: # Assume it's a local file path
    with open(pdf_path, "rb") as pdf_file:
      pdf_reader = PdfReader(pdf_file)
      for page in pdf_reader.pages:
        text += page.extract_text()
  return text

In [6]:
# Setting the parameters for text splitting
sent_text_splitter = SentenceTransformersTokenTextSplitter(
    chunk_overlap=10, # Overlap tokens for context continuity
    model_name='sentence-transformers/all-MiniLM-L6-v2', # Embedding model for tokenization
    tokens_per_chunk=100 # Chunk size (tunable for your use case)
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
# Chunking the text into smaller pieces for better processing and retrieval
def chunk_text(text,file_name):
  chunks = []
  for chunk in sent_text_splitter.split_text(text):
    chunks.append({"content":chunk,
                   "metadata":{"filename":file_name}})
  return chunks

In [14]:
# Initializing the embedding model for converting text chunks into vector representations
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
# Storing the chunks in ChromaDB for efficient retrieval based on semantic similarity
def store_in_chroma(chunks,persist_directory="./chroma_store"):
  texts = [c["content"] for c in chunks]
  metadatas = [c["metadata"] for c in chunks]
  db = Chroma.from_texts(texts, embedding_model,metadatas=metadatas, persist_directory=persist_directory)
  return db

In [16]:
pdf_path = "https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf"
filename = pdf_path.split("/")[-1]
text = extract_pdf_content(pdf_path)
chunks = chunk_text(text,filename)
db = store_in_chroma(chunks)

In [17]:
# Basic retrieval function to get relevant chunks from ChromaDB based on query similarity
def search_chroma(query,db,top_k=5):
  results = db.similarity_search(query,k=top_k)
  chunks = [{"content":d.page_content,"metadata":d.metadata} for d in results]
  return chunks

In [21]:
# --- Configure Gemini API Key ---
# Securely load your Google Gemini API key from Colab userdata.
# Use Case: Keeps credentials safe and enables authenticated LLM access.
import google.generativeai as genai # Correct import for genai.configure
from google.colab import userdata
api_key = userdata.get('G_API')
genai.configure(api_key=api_key)

In [26]:
# --- RAG: Retrieval-Augmented Generation Pipeline ---
# This function implements the classic RAG workflow: retrieve relevant chunks, build a context, and generate an answer using Gemini.
# Use Case: Direct Q&A over documents, with source attribution for transparency.

def rag_answer(query,db,top_k=5):
  # Step 1: Retrieve relevant chunks from ChromaDB
  chunks = search_chroma(query,db,top_k)
  context = "\n\n".join([c["content"] for c in chunks])
  prompt = f"""Use the following context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

  {context}

  Question: {query}
  Answer:"""

  # Step 2: Generate answer using Gemini LLM
  model = genai.GenerativeModel('models/gemini-2.5-flash')
  response = model.generate_content(prompt)
  answer = response.text
  # Step 3: Extract filenames for source transparency
  filenames = [c["metadata"]["filename"] for c in chunks if isinstance(c.get("metadata"), dict) and "filename" in c["metadata"]]
  filenames = set(filenames)
  return answer, filenames

In [28]:
# --- Example: RAG Pipeline in Action ---

rag_answer("What is attention",db)

('An attention function can be described as mapping a query and a set of key - value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.',
 {'3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf'})